## Validación del procedimiento de reconstrucción

El archivo procesado `014_1.npy` proporcionado en el dataset se encuentra dañado y no puede cargarse correctamente. Sin embargo, su versión correspondiente en `d01_raw_data` sí está disponible, por lo que se buscó reconstruir el archivo procesado aplicando las transformaciones descritas para el conjunto de datos.

Antes de reconstruir `014_1.npy`, es necesario comprobar que el procedimiento utilizado realmente reproduce los archivos processed oficiales. Para ello, se seleccionaron archivos válidos de las actividades `000`, `001` y `013`, considerando ambos grupos de sensores (`_1` y `_2`).

Para cada archivo se realizó el siguiente procedimiento:

1. Se cargó su versión original de `d01_raw_data`.
2. Se aplicó un filtro de media móvil con una ventana de 10 puntos.
3. Se restó la media de cada canal dentro de cada muestra.
4. Se obtuvo un nuevo arreglo reconstruido.
5. Se comparó el arreglo reconstruido con su versión oficial de `d02_processed_data`.

Los archivos reconstruidos utilizados en esta comparación se encuentran en la carpeta `comparacion_reconstruidos`, mientras que los archivos procesados oficiales se encuentran en `Rehab_exercise/d02_processed_data`.

La comparación permitirá verificar:

- Que ambos arreglos tengan las mismas dimensiones.
- Si los valores son exactamente iguales.
- Si son numéricamente equivalentes considerando pequeñas diferencias de redondeo.
- La diferencia máxima entre ambos arreglos.
- El error absoluto medio.

La igualdad exacta puede ser falsa debido a diferencias extremadamente pequeñas producidas por las operaciones con números de punto flotante. Por esta razón, la comprobación principal será la igualdad numérica mediante `np.allclose`, utilizando una tolerancia de `1 × 10⁻¹⁰`.

Si la transformación produce resultados numéricamente equivalentes en todos los archivos de referencia, se considerará validado el procedimiento y podrá aplicarse al archivo raw `014_1.npy` para generar una nueva versión procesada.

En este documento compararemos algunos archivos de la carpeta de comparacion_reconstruidos con los originales de la capreta Rehab_ecercise para después poderrecuperar el 014_1.npy

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Carpeta con los archivos processed oficiales
RUTA_ORIGINALES = Path(
    "../Rehab_exercise/d02_processed_data"
)

# Carpeta con las reconstrucciones
RUTA_RECONSTRUIDOS = Path(
    "comparacion_reconstruidos"
)

# Archivos oficiales válidos que pueden compararse
archivos_comparacion = [
    "000_1.npy",
    "000_2.npy",
    "001_1.npy",
    "001_2.npy",
    "013_1.npy",
    "013_2.npy"
]

resultados = []

for nombre in archivos_comparacion:

    original = np.load(
        RUTA_ORIGINALES / nombre,
        allow_pickle=False
    )

    reconstruido = np.load(
        RUTA_RECONSTRUIDOS / nombre,
        allow_pickle=False
    )

    diferencia = np.abs(original - reconstruido)

    resultados.append({
        "archivo": nombre,
        "mismo_shape": original.shape == reconstruido.shape,
        "igualdad_exacta": np.array_equal(
            original,
            reconstruido
        ),
        "igualdad_numerica": np.allclose(
            original,
            reconstruido,
            rtol=1e-10,
            atol=1e-10
        ),
        "diferencia_maxima": diferencia.max(),
        "error_absoluto_medio": diferencia.mean()
    })

df_comparacion = pd.DataFrame(resultados)

display(df_comparacion)

,archivo,mismo_shape,igualdad_exacta,igualdad_numerica,diferencia_maxima,error_absoluto_medio
0,000_1.npy,True,False,True,2.060574e-13,1.206987e-14
1,000_2.npy,True,False,True,4.689582e-13,2.206360e-14
2,001_1.npy,True,False,True,5.329071e-13,1.205184e-14
3,001_2.npy,True,False,True,1.847411e-13,1.606440e-14
4,013_1.npy,True,False,True,8.526513e-13,1.440109e-14
5,013_2.npy,True,False,True,4.405365e-13,2.886373e-14


### Interpretación de la comparación

Todos los archivos reconstruidos conservaron las mismas dimensiones que sus versiones processed oficiales. La igualdad exacta fue falsa porque `np.array_equal` exige que cada valor sea idéntico hasta el último bit.

Sin embargo, la igualdad numérica fue verdadera en todos los casos. Las diferencias máximas se encontraron en el orden de `10⁻¹³` y los errores absolutos medios en el orden de `10⁻¹⁴`. Estas diferencias son considerablemente menores que la tolerancia establecida de `10⁻¹⁰` y corresponden al redondeo normal de operaciones con datos `float64`.

La transformación fue validada en seis archivos correspondientes a tres actividades diferentes y a ambos grupos de sensores. Por lo tanto, existe evidencia suficiente para aplicar el mismo procedimiento al archivo raw `014_1.npy` y generar una versión procesada compatible con el resto del conjunto de datos.

## Verificación del archivo `014_1.npy` reconstruido

Después de validar la transformación con archivos processed oficiales, se aplicó el mismo procedimiento al archivo raw `014_1.npy`.

En este paso se verificará que el archivo reconstruido:

- Pueda cargarse correctamente con NumPy.
- Tenga tres dimensiones.
- Contenga 359 muestras.
- Tenga 880 puntos temporales y 6 canales.
- No contenga valores faltantes o infinitos.
- Tenga la misma estructura que `014_2.npy`.

Esta revisión final permite confirmar que el archivo puede incorporarse a `d02_processed_data` y utilizarse posteriormente junto con `014_2.npy`.

In [2]:
# Archivo 014_1.npy reconstruido.
RUTA_014_1 = Path("resultados_reconstruccion/014_1.npy")

# Archivo 014_2.npy oficial.
RUTA_014_2 = Path(
    "../Rehab_exercise/d02_processed_data/014_2.npy"
)


# ---------------------------------------------------------
# Comprobación de las rutas
# ---------------------------------------------------------

if not RUTA_014_1.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo reconstruido:\n{RUTA_014_1}"
    )

if not RUTA_014_2.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo oficial:\n{RUTA_014_2}"
    )


# ---------------------------------------------------------
# Carga segura de los archivos
# ---------------------------------------------------------

# allow_pickle=False confirma que los archivos contienen
# arreglos numéricos y no objetos serializados.
datos_014_1 = np.load(
    RUTA_014_1,
    allow_pickle=False
)

datos_014_2 = np.load(
    RUTA_014_2,
    allow_pickle=False
)


# ---------------------------------------------------------
# Revisión de dimensiones y tipo de dato
# ---------------------------------------------------------

print("Archivo 014_1 reconstruido")
print("Shape:", datos_014_1.shape)
print("Tipo de dato:", datos_014_1.dtype)

print("\nArchivo 014_2 oficial")
print("Shape:", datos_014_2.shape)
print("Tipo de dato:", datos_014_2.dtype)


# ---------------------------------------------------------
# Búsqueda de valores faltantes e infinitos
# ---------------------------------------------------------

nulos_014_1 = np.isnan(datos_014_1).sum()
infinitos_014_1 = np.isinf(datos_014_1).sum()

nulos_014_2 = np.isnan(datos_014_2).sum()
infinitos_014_2 = np.isinf(datos_014_2).sum()

print("\nValores inválidos")

print("014_1 - valores faltantes:", nulos_014_1)
print("014_1 - valores infinitos:", infinitos_014_1)

print("014_2 - valores faltantes:", nulos_014_2)
print("014_2 - valores infinitos:", infinitos_014_2)


# ---------------------------------------------------------
# Comparación de la estructura
# ---------------------------------------------------------

misma_forma = datos_014_1.shape == datos_014_2.shape
mismo_tipo = datos_014_1.dtype == datos_014_2.dtype

estructura_esperada_014_1 = (
    datos_014_1.ndim == 3
    and datos_014_1.shape[1] == 880
    and datos_014_1.shape[2] == 6
)

estructura_esperada_014_2 = (
    datos_014_2.ndim == 3
    and datos_014_2.shape[1] == 880
    and datos_014_2.shape[2] == 6
)

print("\nComparación de la estructura")

print("Misma forma:", misma_forma)
print("Mismo tipo de dato:", mismo_tipo)
print(
    "014_1 tiene estructura (muestras, 880, 6):",
    estructura_esperada_014_1
)
print(
    "014_2 tiene estructura (muestras, 880, 6):",
    estructura_esperada_014_2
)


# ---------------------------------------------------------
# Validación final
# ---------------------------------------------------------

archivo_014_1_valido = (
    estructura_esperada_014_1
    and nulos_014_1 == 0
    and infinitos_014_1 == 0
)

archivos_compatibles = (
    misma_forma
    and mismo_tipo
    and estructura_esperada_014_1
    and estructura_esperada_014_2
)

print("\nResultado final")

print("014_1 reconstruido es válido:", archivo_014_1_valido)
print("014_1 y 014_2 son compatibles:", archivos_compatibles)


# Detenemos la ejecución si se encuentra algún problema.
if not archivo_014_1_valido:
    raise ValueError(
        "El archivo 014_1 reconstruido no superó la validación."
    )

if not archivos_compatibles:
    raise ValueError(
        "Los archivos 014_1 y 014_2 no tienen estructuras compatibles."
    )

print("\nValidación completada correctamente.")

Archivo 014_1 reconstruido
Shape: (359, 880, 6)
Tipo de dato: float64

Archivo 014_2 oficial
Shape: (359, 880, 6)
Tipo de dato: float64

Valores inválidos
014_1 - valores faltantes: 0
014_1 - valores infinitos: 0
014_2 - valores faltantes: 0
014_2 - valores infinitos: 0

Comparación de la estructura
Misma forma: True
Mismo tipo de dato: True
014_1 tiene estructura (muestras, 880, 6): True
014_2 tiene estructura (muestras, 880, 6): True

Resultado final
014_1 reconstruido es válido: True
014_1 y 014_2 son compatibles: True

Validación completada correctamente.


Con esto pasaremos el `014.1.npyp` y los susituiremos por el archivo dañado que está en la capreta Rehab_exercise